In [32]:
import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

# 1️⃣ Load the dataset
file_path = r"C:\Users\Ashmeet\OneDrive\Documents\11min.csv"
df = pd.read_csv(file_path)

# 2️⃣ Clean column names (remove special characters and spaces)
df.columns = df.columns.str.replace(r'[\[\]° ]', '', regex=True)  # Remove '[', ']', '°', and spaces
print("Cleaned Column Names:", df.columns.tolist())

# 3️⃣ Add label (0 for walking, since we only have walking data for now)
df["label"] = 0  # All data is labeled as walking (0)

# 4️⃣ Drop unnecessary columns (time and temperature)
X = df.drop(columns=["label", "Times", "stts22h_tempC"])  # Use cleaned column names
y = df["label"]

# 5️⃣ Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 6️⃣ Define LightGBM model (binary classification)
model = lgb.LGBMClassifier(
    objective="binary",  # Binary classification (walking vs. not walking)
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    n_estimators=1000,
    random_state=42
)

# 7️⃣ Train the model with early stopping
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="binary_logloss",  # Binary log loss for binary classification
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# 8️⃣ Make predictions
y_pred = model.predict(X_val)

# 9️⃣ Evaluate accuracy
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy:.4f}")

# 🔟 Classification report and confusion matrix
print("\nClassification Report:")
print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

# 1️⃣1️⃣ Feature importance
try:
    # Plot feature importance
    lgb.plot_importance(model, importance_type='gain', max_num_features=10)  # Use 'gain' or 'split'
    plt.title("Feature Importance (LightGBM)")
    plt.show()
except Exception as e:
    print(f"Error plotting feature importance: {e}")
    print("Ensure the model was trained successfully and features are valid.")

# 1️⃣2️⃣ Debugging: Print feature importance values
importance = model.feature_importances_
feature_names = model.feature_name_
print("\nFeature Importance Values:")
for feature, importance_value in zip(feature_names, importance):
    print(f"{feature}: {importance_value}")

Cleaned Column Names: ['Times', 'lsm6dsv16x_acc_xg', 'lsm6dsv16x_acc_yg', 'lsm6dsv16x_acc_zg', 'lsm6dsv16x_gyro_xdps', 'lsm6dsv16x_gyro_ydps', 'lsm6dsv16x_gyro_zdps', 'lis2du12_acc_xg', 'lis2du12_acc_yg', 'lis2du12_acc_zg', 'lis2mdl_mag_xG', 'lis2mdl_mag_yG', 'lis2mdl_mag_zG', 'lps22df_presshPa', 'stts22h_tempC']
[LightGBM] [Warning] Contains only one class
[LightGBM] [Info] Number of positive: 0, number of negative: 838860
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.066571 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3069
[LightGBM] [Info] Number of data points in the train set: 838860, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000000 -> initscore=-34.538776
[LightGBM] [Info] Start training from score -34.538776
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
Training until validation scores don't imp

In [5]:
pip install lightgbm pandas scikit-learn numpy


Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install --upgrade lightgbm


In [ ]:
"C:\Users\Ashmeet\OneDrive\Documents\500Running.csv"
"C:\Users\Ashmeet\OneDrive\Documents\falling1.csv"
"C:\Users\Ashmeet\OneDrive\Documents\falling2.csv"
"C:\Users\Ashmeet\OneDrive\Documents\falling3.csv"
"C:\Users\Ashmeet\OneDrive\Documents\falling4.csv"
"C:\Users\Ashmeet\OneDrive\Documents\11min.csv"

In [5]:
pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load and preprocess data
file_paths = {
    'walking': r"C:\Users\Ashmeet\OneDrive\Documents\11min.csv",
    'running': r"C:\Users\Ashmeet\OneDrive\Documents\500Running.csv",
    'falling_while_running': r"C:\Users\Ashmeet\OneDrive\Documents\falling1.csv",
    'falling_while_walking': r"C:\Users\Ashmeet\OneDrive\Documents\falling3.csv",
}

dataframes = []
for label, path in file_paths.items():
    df = pd.read_csv(path)
    df['label'] = label  # Assign activity label
    dataframes.append(df)

data = pd.concat(dataframes, ignore_index=True).dropna()

# Features and target selection
features = [
    'lsm6dsv16x_acc_x [g]', 'lsm6dsv16x_acc_y [g]', 'lsm6dsv16x_acc_z [g]',
    'lsm6dsv16x_gyro_x [dps]', 'lsm6dsv16x_gyro_y [dps]', 'lsm6dsv16x_gyro_z [dps]',
    'lis2du12_acc_x [g]', 'lis2du12_acc_y [g]', 'lis2du12_acc_z [g]',
    'lis2mdl_mag_x [G]', 'lis2mdl_mag_y [G]', 'lis2mdl_mag_z [G]',
    'lps22df_press [hPa]', 'mp23db01hp_mic [Waveform]', 'stts22h_temp [°C]'
]

X = data[features].values
y = data['label'].values

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Normalize data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshape for CNN input (samples, time steps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

# Select only 20% of the data for training
train_size = int(0.02 * len(X))  # 20% of total dataset
indices = np.random.choice(len(X), train_size, replace=False)  # Randomly select indices

X_train = X[indices]
y_train = y[indices]

# Build CNN model
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(np.unique(y)), activation='softmax')  # Output layer for multi-class classification
])

# Compile model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train model only on 20% of data
model.fit(X_train, y_train, epochs=20, batch_size=32)

# Save trained model
model.save("cnn_activity_recognition_20percent.h5")


In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load and merge datasets
file_paths = {
    'walking': r"C:\Users\Ashmeet\OneDrive\Documents\11min.csv",
    'running': r"C:\Users\Ashmeet\OneDrive\Documents\500Running.csv",
    'falling_while_running': r"C:\Users\Ashmeet\OneDrive\Documents\falling1.csv",
    'falling_while_walking': r"C:\Users\Ashmeet\OneDrive\Documents\falling3.csv",
}


dataframes = []
for label, path in file_paths.items():
    df = pd.read_csv(path)
    df['label'] = label  # Assign activity label
    dataframes.append(df)

data = pd.concat(dataframes, ignore_index=True)

# Check for missing values
data = data.dropna()

# Extract features and target column
features = [
    'lsm6dsv16x_acc_x [g]', 'lsm6dsv16x_acc_y [g]', 'lsm6dsv16x_acc_z [g]',
    'lsm6dsv16x_gyro_x [dps]', 'lsm6dsv16x_gyro_y [dps]', 'lsm6dsv16x_gyro_z [dps]',
    'lis2du12_acc_x [g]', 'lis2du12_acc_y [g]', 'lis2du12_acc_z [g]',
    'lis2mdl_mag_x [G]', 'lis2mdl_mag_y [G]', 'lis2mdl_mag_z [G]',
    'lps22df_press [hPa]', 'mp23db01hp_mic [Waveform]', 'stts22h_temp [°C]'
]
X = data[features]
y = data['label']

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define Optuna objective function for tuning LightGBM
def objective(trial):
    params = {
        'objective': 'multiclass',
        'num_class': len(np.unique(y)),
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),
        'max_bin': trial.suggest_int('max_bin', 255, 512),
        'num_boost_round': trial.suggest_int('num_boost_round', 100, 1000),
    }
    
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    val_scores = []
    
    for train_idx, val_idx in kf.split(X_train, y_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        model = lgb.train(params, train_data, valid_sets=[val_data], callbacks=[lgb.early_stopping(50)])
        
        y_pred = np.argmax(model.predict(X_val), axis=1)
        val_scores.append(accuracy_score(y_val, y_pred))
    
    return np.mean(val_scores)

# Run Optuna for hyperparameter tuning
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

# Train the best model with optimized hyperparameters
best_params = study.best_params
best_params['objective'] = 'multiclass'
best_params['num_class'] = len(np.unique(y))
best_params['metric'] = 'multi_logloss'

train_data = lgb.Dataset(X_train, label=y_train)
model = lgb.train(best_params, train_data)

# Predictions
y_pred = np.argmax(model.predict(X_test), axis=1)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


[I 2025-02-21 17:46:55,698] A new study created in memory with name: no-name-09f2f7ee-d8ab-404f-88e3-e5e18639b67e


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.064806 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5518
[LightGBM] [Info] Number of data points in the train set: 998766, number of used features: 15
[LightGBM] [Info] Start training from score -1.747011
[LightGBM] [Info] Start training from score -1.872175
[LightGBM] [Info] Start training from score -0.397620
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

[W 2025-02-21 17:47:55,993] Trial 0 failed with parameters: {'learning_rate': 0.06406700735799477, 'num_leaves': 22, 'max_depth': 11, 'min_data_in_leaf': 42, 'feature_fraction': 0.5134980426066947, 'bagging_fraction': 0.7793624515308595, 'bagging_freq': 2, 'max_bin': 415, 'num_boost_round': 408} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\Ashmeet\anaconda3\lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "<ipython-input-3-0c2307e5b099>", line 80, in objective
    model = lgb.train(params, train_data, valid_sets=[val_data], callbacks=[lgb.early_stopping(50)])
  File "C:\Users\Ashmeet\anaconda3\lib\site-packages\lightgbm\engine.py", line 322, in train
    booster.update(fobj=fobj)
  File "C:\Users\Ashmeet\anaconda3\lib\site-packages\lightgbm\basic.py", line 4155, in update
    _LIB.LGBM_BoosterUpdateOneIter(
KeyboardInterrupt
[W 2025-02-21 17:47:56,002] Trial 0 faile

KeyboardInterrupt: 